# DEW PE Final-FP Relative-Error Verification

Reconstruct the quantized G16 datapath, block-exponent Eacc asymmetric DEW-ACC (`T_SKIP=9`, `T_REPLACE=3`, 24-bit), and outlier FP path from `input.dat`. Compare the 256 final RTL FP32 results in `output.dat` against the software DEW reference using relative error.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RELATIVE_EPSILON = 1e-12
RELATIVE_TOLERANCE = 1e-3


def find_research1_root() -> Path:
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        candidates = (base, base / 'research1')
        for candidate in candidates:
            if (candidate / 'rtl' / '03_DEW-PE').is_dir() and (candidate / 'experiments' / '10_RTL_Trace').is_dir():
                return candidate
    raise FileNotFoundError('Cannot locate the research1 project root.')


RESEARCH1_ROOT = find_research1_root()
DEW_TESTBED = RESEARCH1_ROOT / 'rtl' / '03_DEW-PE' / '00_TESTBED'
TRACE_DIR = (
    RESEARCH1_ROOT
    / 'experiments'
    / '10_RTL_Trace'
    / 'llama2-7b'
    / 'llama2_layer16_down_proj_g16_wbfp4_t2abie4_dew_tskip9_treplace3'
)
REPORT_DIR = RESEARCH1_ROOT / 'rtl' / '00_verification' / 'reports' / 'dew_pe'
INPUT_PATH = DEW_TESTBED / 'input.dat'
OUTPUT_PATH = DEW_TESTBED / 'output.dat'
METADATA_PATH = TRACE_DIR / 'trace_metadata.json'
INDEX_PATH = TRACE_DIR / 'trace_index.csv'

required_paths = [INPUT_PATH, OUTPUT_PATH, METADATA_PATH, INDEX_PATH]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Missing DEW verification files:\n' + '\n'.join(missing_paths))

metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
trace_index = pd.read_csv(INDEX_PATH)
assert metadata['trace_format'] == 'dew_pe_t2a_bie4_cycle_input_v1'
config = metadata['trace_config']
GROUP_SIZE = int(config['group_size'])
MANTISSA_BITS = int(config['mantissa_bits'])
EXPONENT_BIAS = int(config['exponent_bias'])
T_SKIP = int(config['t_skip'])
T_REPLACE = int(config['t_replace'])
DEW_ACC_WIDTH = int(config['dew_acc_width'])
BFP_MAG_WIDTH = 10
BLOCKS_PER_DOT = int(metadata['blocks_per_dot_product'])
RECORDS_PER_DOT = int(metadata['records_per_dot_product'])
DOT_PRODUCT_COUNT = int(metadata['dot_product_count'])
LOGICAL_RECORDS = int(metadata['logical_records'])

assert (GROUP_SIZE, MANTISSA_BITS, EXPONENT_BIAS) == (16, 3, 15)
assert (T_SKIP, T_REPLACE, DEW_ACC_WIDTH) == (9, 3, 24)
print({
    'input': str(INPUT_PATH),
    'output': str(OUTPUT_PATH),
    'dot_products': DOT_PRODUCT_COUNT,
    'relative_tolerance': RELATIVE_TOLERANCE,
})

## Parse and validate `input.dat` and `output.dat`

In [ ]:
FIELD_NAMES = [
    'acc_clear', 'weight_load', 'in_valid',
    'w_sign', 'w_exp', 'w_magnitude',
    'a_sign', 'a_exp', 'oa_exp', 'a_magnitude', 'oi1', 'oi2',
]
FIELD_PATTERNS = {
    'acc_clear': r'[01]',
    'weight_load': r'[01]',
    'in_valid': r'[01]',
    'w_sign': r'[0-9a-fA-F]{4}',
    'w_exp': r'[0-9a-fA-F]{2}',
    'w_magnitude': r'[0-9a-fA-F]{12}',
    'a_sign': r'[0-9a-fA-F]{4}',
    'a_exp': r'[0-9a-fA-F]{2}',
    'oa_exp': r'[0-9a-fA-F]{2}',
    'a_magnitude': r'[0-9a-fA-F]{12}',
    'oi1': r'[0-9a-fA-F]',
    'oi2': r'[0-9a-fA-F]',
}

stimulus_hex = pd.read_csv(
    INPUT_PATH, sep=r'\s+', names=FIELD_NAMES, dtype=str, engine='python'
)
rtl_output_hex = pd.read_csv(
    OUTPUT_PATH, header=None, names=['word'], dtype=str
)['word'].str.strip()

assert len(stimulus_hex) == LOGICAL_RECORDS
assert len(rtl_output_hex) == DOT_PRODUCT_COUNT
assert len(trace_index) == DOT_PRODUCT_COUNT
for field, pattern in FIELD_PATTERNS.items():
    if not stimulus_hex[field].str.fullmatch(pattern).all():
        raise ValueError(f'Malformed {field} field in input.dat')
if not rtl_output_hex.str.fullmatch(r'[0-9a-fA-F]{8}').all():
    raise ValueError('Malformed 32-bit word in output.dat')

stimulus = stimulus_hex.copy()
for field in FIELD_NAMES:
    stimulus[field] = stimulus[field].map(lambda value: int(value, 16))
records = stimulus.to_numpy(dtype=np.int64)
position = np.arange(LOGICAL_RECORDS, dtype=np.int64) % RECORDS_PER_DOT
np.testing.assert_array_equal(records[:, 0].astype(bool), position == 0)
np.testing.assert_array_equal(records[:, 1].astype(bool), position < RECORDS_PER_DOT-1)
np.testing.assert_array_equal(records[:, 2].astype(bool), position != 0)
setup_records = np.arange(DOT_PRODUCT_COUNT, dtype=np.int64) * RECORDS_PER_DOT
np.testing.assert_array_equal(trace_index['setup_record'].to_numpy(), setup_records)
np.testing.assert_array_equal(trace_index['final_record'].to_numpy(), setup_records + RECORDS_PER_DOT-1)
np.testing.assert_array_equal(records[setup_records, 10], np.full(DOT_PRODUCT_COUNT, 15))
np.testing.assert_array_equal(records[setup_records, 11], np.full(DOT_PRODUCT_COUNT, 15))
print(f'[PASS] Parsed {LOGICAL_RECORDS:,} records and {DOT_PRODUCT_COUNT} final FP32 results.')

## Reconstruct the quantized DEW datapath

In [ ]:
def unpack_sign(word: int) -> np.ndarray:
    return np.fromiter(
        ((word >> lane) & 1 for lane in range(GROUP_SIZE)),
        dtype=np.int64, count=GROUP_SIZE,
    )


def unpack_magnitude(word: int) -> np.ndarray:
    mask = (1 << MANTISSA_BITS) - 1
    return np.fromiter(
        ((word >> (lane * MANTISSA_BITS)) & mask for lane in range(GROUP_SIZE)),
        dtype=np.int64, count=GROUP_SIZE,
    )


def signed_products(w_sign: int, w_magnitude: int, a_sign: int, a_magnitude: int) -> np.ndarray:
    products = unpack_magnitude(w_magnitude) * unpack_magnitude(a_magnitude)
    signs = unpack_sign(w_sign) ^ unpack_sign(a_sign)
    return np.where(signs != 0, -products, products)


def decode_fp32_words(words: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    bits = np.fromiter((int(word, 16) for word in words), dtype=np.uint32, count=len(words))
    return bits, bits.view(np.float32).astype(np.float64)


def decode_outlier_indices(oi1: int, oi2: int) -> list[int]:
    if oi1 == 15 and oi2 == 15:
        return []
    if oi2 == 0:
        return [oi1]
    if oi1 == oi2:
        raise ValueError(f'Duplicate two-outlier index: ({oi1:x}, {oi2:x})')
    return [oi1, oi2]


def leading_index(value: int) -> int:
    return abs(value).bit_length() - 1


def shift_toward_zero(value: int, amount: int) -> int:
    if amount == 0:
        return value
    return (1 if value >= 0 else -1) * (abs(value) >> amount)


def format_dew_spill(value: int, block_exp: int) -> float:
    magnitude = abs(value)
    if magnitude == 0:
        return 0.0
    shift = max(0, leading_index(magnitude) - (BFP_MAG_WIDTH - 1))
    formatted_magnitude = magnitude >> shift
    signed_magnitude = -formatted_magnitude if value < 0 else formatted_magnitude
    return np.ldexp(float(signed_magnitude), block_exp + shift - EXPONENT_BIAS)


software_reference = np.zeros(DOT_PRODUCT_COUNT, dtype=np.float64)
ideal_quantized_reference = np.zeros(DOT_PRODUCT_COUNT, dtype=np.float64)
fp_term_count = np.zeros(DOT_PRODUCT_COUNT, dtype=np.int64)
skip_count = np.zeros(DOT_PRODUCT_COUNT, dtype=np.int64)
replace_count = np.zeros(DOT_PRODUCT_COUNT, dtype=np.int64)
overflow_count = np.zeros(DOT_PRODUCT_COUNT, dtype=np.int64)
record_index = 0

for dot_index in range(DOT_PRODUCT_COUNT):
    setup = records[record_index]
    record_index += 1
    weight_sign_reg = int(setup[3])
    weight_exp_reg = int(setup[4])
    weight_magnitude_reg = int(setup[5])
    eacc_valid = False
    eacc_exp = 0
    acc_value = 0
    fp_terms = []
    ideal_value = 0.0

    for block_index in range(BLOCKS_PER_DOT):
        row = records[record_index]
        record_index += 1
        (_, weight_load, _, next_w_sign, next_w_exp, next_w_magnitude,
         a_sign, a_exp, oa_exp, a_magnitude, oi1, oi2) = row
        products = signed_products(
            weight_sign_reg, weight_magnitude_reg, int(a_sign), int(a_magnitude)
        )
        outlier_indices = decode_outlier_indices(int(oi1), int(oi2))
        normal_mask = np.ones(GROUP_SIZE, dtype=bool)
        normal_mask[outlier_indices] = False
        normal_partial = int(products[normal_mask].sum(dtype=np.int64))
        outlier_partial = int(products[outlier_indices].sum(dtype=np.int64)) if outlier_indices else 0
        normal_block_exp = weight_exp_reg + int(a_exp) - EXPONENT_BIAS
        outlier_block_exp = weight_exp_reg + int(oa_exp) - EXPONENT_BIAS

        ideal_value += np.ldexp(float(normal_partial), normal_block_exp - EXPONENT_BIAS)
        ideal_value += np.ldexp(float(outlier_partial), outlier_block_exp - EXPONENT_BIAS)
        if outlier_partial != 0:
            fp_terms.append(np.ldexp(float(outlier_partial), outlier_block_exp - EXPONENT_BIAS))

        if normal_partial != 0:
            if not eacc_valid:
                eacc_valid = True
                eacc_exp = normal_block_exp
                acc_value = normal_partial
            else:
                delta = normal_block_exp - eacc_exp
                if delta <= -T_SKIP:
                    skip_count[dot_index] += 1
                elif delta >= T_REPLACE:
                    replace_count[dot_index] += 1
                    eacc_exp = normal_block_exp
                    acc_value = normal_partial
                else:
                    if delta < 0:
                        shifted = shift_toward_zero(normal_partial, -delta)
                        bypass = acc_value
                        common_exp = eacc_exp
                    else:
                        shifted = shift_toward_zero(acc_value, delta)
                        bypass = normal_partial
                        common_exp = normal_block_exp
                    candidate = shifted + bypass
                    minimum = -(1 << (DEW_ACC_WIDTH - 1))
                    maximum = (1 << (DEW_ACC_WIDTH - 1)) - 1
                    if candidate < minimum or candidate > maximum:
                        fp_terms.append(format_dew_spill(candidate, common_exp))
                        overflow_count[dot_index] += 1
                        eacc_valid = False
                        eacc_exp = 0
                        acc_value = 0
                    else:
                        eacc_exp = common_exp
                        acc_value = candidate

        if block_index == BLOCKS_PER_DOT - 1:
            if acc_value != 0:
                fp_terms.append(format_dew_spill(acc_value, eacc_exp))
            eacc_valid = False
            eacc_exp = 0
            acc_value = 0

        if weight_load:
            weight_sign_reg = int(next_w_sign)
            weight_exp_reg = int(next_w_exp)
            weight_magnitude_reg = int(next_w_magnitude)

    fp_accumulator = np.float32(0.0)
    for term in fp_terms:
        fp_accumulator = np.float32(fp_accumulator + np.float32(term))
    software_reference[dot_index] = float(fp_accumulator)
    ideal_quantized_reference[dot_index] = ideal_value
    fp_term_count[dot_index] = len(fp_terms)

assert record_index == LOGICAL_RECORDS
rtl_bits, rtl_result = decode_fp32_words(rtl_output_hex)
print({
    'rtl_range': [float(rtl_result.min()), float(rtl_result.max())],
    'software_reference_range': [float(software_reference.min()), float(software_reference.max())],
    'total_fp_terms': int(fp_term_count.sum()),
    'total_skips': int(skip_count.sum()),
    'total_replacements': int(replace_count.sum()),
    'total_overflows': int(overflow_count.sum()),
})

## Compute relative error and export reports

In [ ]:
def error_metrics(actual: np.ndarray, reference: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    absolute = np.abs(actual - reference)
    relative = absolute / np.maximum(np.abs(reference), RELATIVE_EPSILON)
    return absolute, relative


dew_absolute_error, dew_relative_error = error_metrics(rtl_result, software_reference)
ideal_absolute_error, ideal_relative_error = error_metrics(rtl_result, ideal_quantized_reference)
within_tolerance = dew_relative_error <= RELATIVE_TOLERANCE

comparison = trace_index.copy()
comparison['rtl_hex'] = rtl_output_hex.to_numpy()
comparison['rtl_result'] = rtl_result
comparison['dew_software_reference'] = software_reference
comparison['dew_absolute_error'] = dew_absolute_error
comparison['dew_relative_error'] = dew_relative_error
comparison['within_tolerance'] = within_tolerance
comparison['ideal_quantized_reference'] = ideal_quantized_reference
comparison['ideal_relative_error'] = ideal_relative_error
comparison['fp_term_count'] = fp_term_count
comparison['skip_count'] = skip_count
comparison['replace_count'] = replace_count
comparison['overflow_count'] = overflow_count

summary = {
    'trace_format': metadata['trace_format'],
    'dot_product_count': DOT_PRODUCT_COUNT,
    'relative_tolerance': RELATIVE_TOLERANCE,
    'within_tolerance': int(np.count_nonzero(within_tolerance)),
    'max_relative_error': float(dew_relative_error.max()),
    'mean_relative_error': float(dew_relative_error.mean()),
    'median_relative_error': float(np.median(dew_relative_error)),
    'p95_relative_error': float(np.percentile(dew_relative_error, 95)),
    'p99_relative_error': float(np.percentile(dew_relative_error, 99)),
    'max_absolute_error': float(dew_absolute_error.max()),
    'ideal_quantized_dot_product_error': {
        'max_relative_error': float(ideal_relative_error.max()),
        'mean_relative_error': float(ideal_relative_error.mean()),
        'median_relative_error': float(np.median(ideal_relative_error)),
    },
    'dewa_events': {
        'fp_terms': int(fp_term_count.sum()),
        'skips': int(skip_count.sum()),
        'replacements': int(replace_count.sum()),
        'overflows': int(overflow_count.sum()),
    },
}

REPORT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(REPORT_DIR / 'dew_pe_final_fp_comparison.csv', index=False)
(REPORT_DIR / 'dew_pe_verification_summary.json').write_text(
    json.dumps(summary, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(summary, indent=2))
print('\nWorst DEW RTL results:')
print(comparison.nlargest(10, 'dew_relative_error').to_string(index=False))

## Relative-error overview

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].semilogy(np.maximum(dew_relative_error, 1e-18), marker='.', linestyle='none')
axes[0].axhline(RELATIVE_TOLERANCE, color='red', linestyle='--', label='Tolerance')
axes[0].set_title('RTL vs DEW software reference')
axes[0].set_xlabel('Dot-product index')
axes[0].set_ylabel('Relative error')
axes[0].grid(True, which='both', alpha=0.3)
axes[0].legend()
axes[1].hist(ideal_relative_error, bins=40)
axes[1].set_title('RTL vs ideal quantized dot product')
axes[1].set_xlabel('Relative error')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3)
figure.tight_layout()
plt.show()

## Final status

In [ ]:
if not np.all(within_tolerance):
    failed = int(np.count_nonzero(~within_tolerance))
    raise AssertionError(
        f'DEW PE verification failed: {failed}/{DOT_PRODUCT_COUNT} results exceed '
        f'{RELATIVE_TOLERANCE:.1e}; max relative error={dew_relative_error.max():.6e}.'
    )

print(
    f'[PASS] DEW PE: {DOT_PRODUCT_COUNT}/{DOT_PRODUCT_COUNT} final FP results are within '
    f'{RELATIVE_TOLERANCE:.1e}; max relative error={dew_relative_error.max():.6e}.'
)
print(f'Reports: {REPORT_DIR.resolve()}')